# CV5570 Assignment 2: Object Detection on ACID

This notebook prepares the ACID dataset, trains Faster R-CNN, RetinaNet, and RT-DETR-L, and saves their validation results to Google Drive. The report is made from the saved outputs in the repository.


In [ ]:
# Install packages not already included in the Colab runtime.
%pip -q install ultralytics Pillow PyYAML reportlab
import torch, torchvision, ultralytics
print('PyTorch:', torch.__version__)
print('TorchVision:', torchvision.__version__)
print('Ultralytics:', ultralytics.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Select Runtime > Change runtime type > GPU before training.')

In [ ]:
# Mount Drive, set the shared workspace path, and get the repository code.
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')

# Change this one path if your Google Drive workspace folder is different.
DRIVE_BASE = Path('/content/drive/MyDrive/Assignment 2 CV5570')
DATA_DIR = DRIVE_BASE / 'ACID_3classes' / '3classes'
RUN_DIR = DRIVE_BASE / 'assignment2_run'
ANALYSIS_DIR = RUN_DIR / 'outputs' / 'dataset_analysis'
MODEL_DIR = RUN_DIR / 'outputs' / 'model_runs'
SPLIT_DIR = ANALYSIS_DIR / 'splits'

REPO_URL = 'https://github.com/wick1-22/Assignment-2-CV5570.git'
REPO_DIR = Path('/content/Assignment-2-CV5570')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

print('Repository:', REPO_DIR)
print('Drive workspace:', DRIVE_BASE)
print('Dataset folder:', DATA_DIR)
print('Saved runs:', RUN_DIR)

In [ ]:
# Check the paths created from DRIVE_BASE above.
ANNOTATIONS = DATA_DIR / '3classes.json'
IMAGE_DIR = DATA_DIR
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not ANNOTATIONS.exists():
    raise FileNotFoundError(f'Could not find ACID COCO annotations: {ANNOTATIONS}')
print('Images:', IMAGE_DIR)
print('Annotations:', ANNOTATIONS)
print('Persistent outputs:', RUN_DIR)

## Dataset acquisition and preprocessing

The preprocessing script checks the COCO annotation file and referenced images, counts objects by class, and creates a repeatable 70/20/10 train, validation, and test split using seed 42. It also saves the class-frequency chart, bounding-box distributions, and six images with ground-truth boxes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(REPO_DIR / 'src' / 'data_tools.py'),
                '--annotations', str(ANNOTATIONS), '--images', str(IMAGE_DIR),
                '--output', str(ANALYSIS_DIR), '--seed', '42'], check=True)

## Training and evaluation

Each detector starts from its specified pretrained checkpoint and is trained separately. The saved validation split is used to calculate AP at IoU 0.5 and across IoU 0.50 to 0.95. The test split is held out. Separate model cells preserve results independently if a runtime ends during a later experiment.

In [ ]:
import subprocess, sys
def run_streaming(cmd):
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise RuntimeError(f'Command failed with exit code {code}; see the error output above.')
EPOCHS = 10
BATCH_SIZE = 2
IMAGE_SIZE = 640
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

### Faster R-CNN checkpoint recovery

If the saved Faster R-CNN checkpoint is available, this cell creates the six validation examples. It preserves existing metrics and does not train the model again.

In [ ]:
faster_examples = MODEL_DIR / 'faster_rcnn' / 'qualitative_examples.png'
faster_checkpoint = MODEL_DIR / 'faster_rcnn' / 'best.pth'
if not faster_examples.exists() and faster_checkpoint.exists():
    cmd = [sys.executable, '-u', str(REPO_DIR / 'src' / 'train_models.py'),
           '--annotations', str(ANNOTATIONS), '--images', str(IMAGE_DIR),
           '--splits', str(SPLIT_DIR), '--analysis', str(ANALYSIS_DIR),
           '--output', str(MODEL_DIR), '--image-size', str(IMAGE_SIZE), '--device', DEVICE,
           '--model', 'faster_rcnn', '--examples-only']
    run_streaming(cmd)
else:
    print('Examples already exist or best.pth is missing.')

### Two-stage detector: Faster R-CNN

Faster R-CNN proposes object regions and then classifies and refines each proposal. The experiment uses the TorchVision COCO-pretrained ResNet-50-FPN v2 checkpoint.

In [ ]:
cmd = [sys.executable, '-u', str(REPO_DIR / 'src' / 'train_models.py'),
       '--annotations', str(ANNOTATIONS), '--images', str(IMAGE_DIR),
       '--splits', str(SPLIT_DIR), '--analysis', str(ANALYSIS_DIR),
       '--output', str(MODEL_DIR), '--staging', '/content/acid_yolo_dataset',
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--image-size', str(IMAGE_SIZE), '--device', DEVICE, '--seed', '42',
       '--model', 'faster_rcnn', '--resume']
print('Starting:', ' '.join(cmd), flush=True)
run_streaming(cmd)

### One-stage detector: RetinaNet

RetinaNet predicts class scores and box offsets in one pass. Focal loss reduces the influence of easy background locations. The experiment uses the TorchVision COCO-pretrained ResNet-50-FPN v2 checkpoint.

In [ ]:
cmd = [sys.executable, '-u', str(REPO_DIR / 'src' / 'train_models.py'),
       '--annotations', str(ANNOTATIONS), '--images', str(IMAGE_DIR),
       '--splits', str(SPLIT_DIR), '--analysis', str(ANALYSIS_DIR),
       '--output', str(MODEL_DIR), '--staging', '/content/acid_yolo_dataset',
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--image-size', str(IMAGE_SIZE), '--device', DEVICE, '--seed', '42',
       '--model', 'retinanet', '--resume']
print('Starting:', ' '.join(cmd), flush=True)
run_streaming(cmd)

### Transformer detector: RT-DETR-L

RT-DETR-L uses a transformer encoder and decoder for object detection. The experiment starts from the pretrained `rtdetr-l` checkpoint and records training progress and validation results.

In [ ]:
cmd = [sys.executable, '-u', str(REPO_DIR / 'src' / 'train_models.py'),
       '--annotations', str(ANNOTATIONS), '--images', str(IMAGE_DIR),
       '--splits', str(SPLIT_DIR), '--analysis', str(ANALYSIS_DIR),
       '--output', str(MODEL_DIR), '--staging', '/content/acid_yolo_dataset',
       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
       '--image-size', str(IMAGE_SIZE), '--device', DEVICE, '--seed', '42',
       '--model', 'rtdetr', '--resume']
print('Starting:', ' '.join(cmd), flush=True)
run_streaming(cmd)

## Saved outputs
Model metrics, plots, loss histories, and detection examples are saved in the run folder. The report is assembled from these files after they are copied into the repository.